In [ ]:
import re
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, col
from pyspark.sql.types import StringType

In [ ]:
# Khởi tạo Spark Session cho PoC
spark = SparkSession.builder \
    .appName("Vectorized_PII_Redaction_PoC") \
    .master("local[*]") \
    .getOrCreate()

In [ ]:
# Giả lập data (ở scale 1 Tỷ requests/ngày, tốc độ xử lý là bài toán hóc búa nhất)
data = [
    ("user_123", "My credit card is 1234-5678-9012-3456 and email is test@email.com"),
    ("user_456", "Please summarize this document. Call me at 090-123-4567"),
    ("user_789", "What is the capital of France? No PII here.")
] * 1000  # Nhân bản để test batch
df = spark.createDataFrame(data, ["tenant_id", "prompt"])

In [ ]:
# Compile regex trước ở mức global để tăng tốc
CREDIT_CARD_RE = re.compile(r'\b(?:\d[ -]*?){13,16}\b')
EMAIL_RE = re.compile(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b')

In [ ]:
# SỬ DỤNG PANDAS UDF (Apache Arrow) thay vì Standard UDF
# Lý do (Tradeoff): Standard Python UDF quá chậm cho 1 tỷ records/ngày.
# Pandas UDF (Vectorized) cho phép xử lý batch nhanh gấp 10-100 lần.
@pandas_udf(StringType())
def vectorized_redact_pii(prompts: pd.Series) -> pd.Series:
    # Thao tác trên Pandas Series thay vì từng dòng riêng lẻ
    redacted = prompts.str.replace(CREDIT_CARD_RE, '[CREDIT_CARD]', regex=True)
    redacted = redacted.str.replace(EMAIL_RE, '[EMAIL]', regex=True)
    return redacted

In [ ]:
# Áp dụng hàm
bronze_df = df.withColumn("prompt_redacted", vectorized_redact_pii(col("prompt")))

In [ ]:
print("--- Sample Bronze Data (Redacted with Vectorized UDF) ---")
bronze_df.select("tenant_id", "prompt_redacted").limit(3).show(truncate=False)

In [ ]:
spark.stop()